# APS360 — Transcript Classifier: Colab Training

> **Before running:** Enable GPU via `Runtime → Change runtime type → T4 GPU`

## 1. Clone repo & install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/romeluis/APS360-TranscriptClassifier.git"
REPO_DIR = "/content/APS360-TranscriptClassifier"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print("Repo already cloned — pulled latest changes.")

%cd {REPO_DIR}

In [ ]:
# Python packages
!pip install -q -r requirements.txt
!pip install -q pytorch-crf    # CRF decoder for TranscriptNERModel

# WeasyPrint system libs — required to render HTML → PDF for training data
!apt-get install -q -y libpango-1.0-0 libpangoft2-1.0-0 libharfbuzz0b 2>/dev/null

import torch, multiprocessing
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU:    {gpu_name}")
    print(f"VRAM:   {vram_gb:.1f} GB")
    # Recommend batch size based on GPU
    if vram_gb >= 35:
        print("  → A100 detected: BATCH_SIZE=48 is set in model/config.py")
    else:
        print("  → T4 detected: consider setting BATCH_SIZE=16 in model/config.py")
else:
    print("WARNING: No GPU found — training will be very slow!")
print(f"CPUs:   {multiprocessing.cpu_count()}")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR  = "/content/drive/MyDrive/APS360_checkpoints"
DRIVE_ZIP  = f"{DRIVE_DIR}/output.zip"   # training data cached as a single zip
!mkdir -p {DRIVE_DIR}

## 3. Generate training data (or restore from Drive)

**Pipeline:** HTML template → WeasyPrint → PDF → pypdf text → character-aligned BIO labels

- Checks Drive for `output.zip` first — if found, extracts it (~1 min) instead of regenerating (~55 min)
- After generation, zips the output folder and saves it to Drive as `output.zip`
- `--count 200`: 61 templates × 200 = **12,200 training samples**
- `--workers 2`: matches Colab's 2 CPU cores

In [ ]:
import glob, os, zipfile

COUNT_PER_TEMPLATE = 200
N_TEMPLATES = len(glob.glob("transcript_generator/templates/*.html"))
EXPECTED = N_TEMPLATES * COUNT_PER_TEMPLATE
OUT_DIR = "transcript_generator/output"

def _count_local():
    return len(glob.glob(f"{OUT_DIR}/*/*.json"))

def _extract_zip():
    """Extract DRIVE_ZIP into transcript_generator/ (recreates output/ folder)."""
    print(f"Extracting {DRIVE_ZIP} ...")
    with zipfile.ZipFile(DRIVE_ZIP, "r") as zf:
        total = len(zf.namelist())
        zf.extractall("transcript_generator/")
    extracted = _count_local()
    print(f"✓ Extracted {extracted} samples from output.zip")
    return extracted

def _save_zip():
    """Zip transcript_generator/output/ and save to DRIVE_ZIP."""
    print(f"Zipping {OUT_DIR}/ → {DRIVE_ZIP} ...")
    with zipfile.ZipFile(DRIVE_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(OUT_DIR):
            for fname in files:
                fpath = os.path.join(root, fname)
                # Store as output/<template>/<file> (relative to transcript_generator/)
                arcname = os.path.relpath(fpath, "transcript_generator/")
                zf.write(fpath, arcname)
    size_mb = os.path.getsize(DRIVE_ZIP) / 1e6
    print(f"✓ Saved output.zip ({size_mb:.0f} MB) to Drive")

# ── Main logic ──────────────────────────────────────────────────────────────
print(f"Templates: {N_TEMPLATES}  |  Expected samples: {EXPECTED}")

existing = _count_local()
print(f"Existing local JSON files: {existing}")

if existing >= EXPECTED:
    print("✓ Data already present in runtime — skipping.")

elif os.path.isfile(DRIVE_ZIP):
    print(f"Found {DRIVE_ZIP} — restoring from zip...")
    restored = _extract_zip()
    if restored < EXPECTED:
        print(f"Zip incomplete ({restored} < {EXPECTED}) — regenerating...")
        get_ipython().system(f"""python transcript_generator/main.py \
            --all-templates \
            --count {COUNT_PER_TEMPLATE} \
            --pdf-extract \
            --augment-noise \
            --no-pdf \
            --workers 2""")
        _save_zip()

else:
    print(f"No output.zip found — generating {EXPECTED} transcripts (~55 min)...")
    get_ipython().system(f"""python transcript_generator/main.py \
        --all-templates \
        --count {COUNT_PER_TEMPLATE} \
        --pdf-extract \
        --augment-noise \
        --no-pdf \
        --workers 2""")
    _save_zip()


## 4. Train

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from model.train import train

history = train(
    data_dir="transcript_generator/output",
    output_dir="model/checkpoints",
    # use_fp16 is ignored — train.py auto-detects bf16 support (A100/H100)
    # and falls back to fp32 on older GPUs (T4, V100).
)

## 5. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(epochs, history["train_loss"], "o-", label="train")
axes[0].plot(epochs, history["val_loss"],   "s-", label="val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history["val_token_acc"], "o-", color="royalblue")
axes[1].set_title("Val Token Accuracy"); axes[1].grid(alpha=0.3)

axes[2].plot(epochs, history["val_entity_f1"], "o-", color="green")
axes[2].set_title("Val Entity F1"); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_history.png", dpi=150)
plt.show()

## 6. Save checkpoint to Google Drive

In [ ]:
!rm -rf {DRIVE_DIR}/best
!cp -r model/checkpoints/best              {DRIVE_DIR}/
!cp model/checkpoints/split_info.json      {DRIVE_DIR}/ 2>/dev/null || true
!cp model/logs/history.json                {DRIVE_DIR}/ 2>/dev/null || true
!cp training_history.png                   {DRIVE_DIR}/ 2>/dev/null || true

print(f"✓ Checkpoint saved to {DRIVE_DIR}")

## 7. (Optional) Download checkpoint directly

Alternative to Drive — zip and download straight to your Mac.

In [ ]:
!zip -r bert_checkpoint.zip model/checkpoints/best model/logs/history.json training_history.png

from google.colab import files
files.download("bert_checkpoint.zip")

## 8. (Optional) Smoke test on a test-split sample

In [ ]:
import json, glob
from model.predict import BertNERPredictor
from evaluation.reconstructor import reconstruct_semesters

with open("model/checkpoints/split_info.json") as f:
    split_info = json.load(f)

test_template = split_info["test_templates"][0]
sample_path = sorted(glob.glob(f"transcript_generator/output/{test_template}/*.json"))[0]
with open(sample_path) as f:
    sample = json.load(f)

predictor = BertNERPredictor(checkpoint_dir="model/checkpoints/best")
tags = predictor.predict(sample["tokens"])
semesters = reconstruct_semesters(sample["tokens"], tags)

print(f"Template: {test_template}\n")
for sem in semesters:
    print(f"  {sem['semester_name']}")
    for c in sem["courses"]:
        print(f"    {c['code']:<12} {c['name']:<40} {c['grade']}")